## 1. RRUFF dataset

Get an idea of how the dataset looks.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

# Go up to the root folder, then go into the src folder
sys.path.append(os.path.abspath(os.path.join("..", "src")))

# This will now find mineral_id inside src/
from mineral_id.io import read_rruff_metadata, read_rruff_spectrum


# Identify project root, even when running from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rruff"
    / "excellent_unoriented"
)


print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

In [ ]:
# Find all text files recursively
all_files = sorted(DATA_DIR.rglob("*.txt"))

print(f"Total text files found: {len(all_files)}")

# Select processed Raman spectra only
processed_files = [
    file for file in all_files
    if "Raman_Data_Processed" in file.name
]

print(f"Processed Raman spectra: {len(processed_files)}")

# Display examples
for file in processed_files[:5]:
    print(file.name)

In [ ]:
records = []

for file in processed_files:

    parts = file.name.split("__")

    if len(parts) < 2:
        continue

    mineral = parts[0]
    rruff_id = parts[1]

    records.append({
        "mineral": mineral,
        "rruff_id": rruff_id,
        "filename": file.name,
        "path": str(file)
    })

catalog = pd.DataFrame(records)

print(f"Number of valid records: {len(catalog)}")

display(catalog.head())

In [ ]:
mineral_summary = (
    catalog
    .groupby("mineral")
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique")
    )
    .sort_values("n_specimens", ascending=False)
)
print(f"Number of unique minerals: {len(mineral_summary)}")
print(f"Number of unique specimens: {catalog['rruff_id'].nunique()}")
print(f"Number of unique spectra: {len(catalog)}")
display(mineral_summary.head(25))

In [ ]:
# Select the first available spectrum
sample_file = Path(catalog.iloc[0]["path"])

print("Filename:")
print(sample_file.name)

print("\nFirst 25 lines of the file:")

with open(sample_file, "r", errors="replace") as f:
    for i in range(25):
        line = f.readline()

        if not line:
            break

        print(line.rstrip())

In [ ]:
# Select the first available spectrum
sample_file = Path(catalog[catalog["mineral"] == "unknown"].iloc[0]["path"])

print("Filename:")
print(sample_file.name)

print("\nFirst 25 lines of the file:")

with open(sample_file, "r", errors="replace") as f:
    for i in range(25):
        line = f.readline()

        if not line:
            break

        print(line.rstrip())

## 2. Inspecting individual Raman spectra

In this section, we inspect the metadata and numerical
structure of individual RRUFF Raman files.

The objective is to understand the spectral representation
before designing the preprocessing pipeline.

In [ ]:
# Select our first spectrum
sample_path = Path(catalog.iloc[0]["path"])

# Load metadata and numerical data
sample_metadata = read_rruff_metadata(sample_path)

sample_spectrum = read_rruff_spectrum(sample_path)

# Display important metadata
for key in [
    "NAMES",
    "RRUFFID",
    "IDEAL CHEMISTRY",
    "RAMAN WAVELENGTH",
    "FILETYPE"
]:
    print(f"{key}: {sample_metadata.get(key)}")

# Display first observations
display(sample_spectrum.head(10))

# Basic information
print("\nNumber of spectral points:", len(sample_spectrum))

print(
    "Raman shift range:",
    sample_spectrum["raman_shift_cm1"].min(),
    "to",
    sample_spectrum["raman_shift_cm1"].max(),
    "cm^-1"
)

print(
    "Intensity range:",
    sample_spectrum["intensity"].min(),
    "to",
    sample_spectrum["intensity"].max()
)

In [ ]:
x = sample_spectrum["raman_shift_cm1"].to_numpy()
y = sample_spectrum["intensity"].to_numpy()

# Check that the Raman shift is strictly increasing
strictly_increasing = np.all(np.diff(x) > 0)

# Check duplicate Raman shift values
duplicate_shifts = sample_spectrum[
    "raman_shift_cm1"
].duplicated().sum()

# Check for missing and infinite values
finite_values = np.isfinite(x).all() and np.isfinite(y).all()

print("Strictly increasing Raman shift:", strictly_increasing)
print("Duplicate Raman shifts:", duplicate_shifts)
print("All values finite:", finite_values)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    sample_spectrum["raman_shift_cm1"],
    sample_spectrum["intensity"],
    linewidth=1
)

plt.xlabel("Raman shift (cm$^{-1}$)")
plt.ylabel("Processed intensity")
plt.title(
    f"Raman spectrum: {sample_metadata.get('NAMES')}"
)

plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Number of independent RRUFF IDs per mineral
specimens_per_mineral = (
    catalog.groupby("mineral")["rruff_id"]
    .nunique()
    .sort_values(ascending=False)
)

# Count classes meeting different minimum requirements
thresholds = [5, 8, 10, 12, 15, 20]

for threshold in thresholds:

    n_classes = (specimens_per_mineral >= threshold).sum()

    print(
        f"Classes with at least {threshold} specimens: "
        f"{n_classes}"
    )

In [ ]:
# Extract laser wavelength from filenames
catalog["laser_nm"] = catalog["filename"].str.extract(
    r"__Raman__(\d+)__"
)

# Number of spectra, specimens and minerals per wavelength
laser_summary = (
    catalog
    .groupby("laser_nm", dropna=False)
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique"),
        n_minerals=("mineral", "nunique")
    )
    .sort_values("n_spectra", ascending=False)
)

display(laser_summary)

In [ ]:
# Check whether an RRUFF ID appears with multiple mineral labels
labels_per_id = (
    catalog.groupby("rruff_id")["mineral"]
    .nunique()
)

conflicting_ids = labels_per_id[labels_per_id > 1]

print("Total unique RRUFF IDs:", len(labels_per_id))
print("IDs associated with multiple mineral labels:", len(conflicting_ids))

if len(conflicting_ids) > 0:
    display(
        catalog[
            catalog["rruff_id"].isin(conflicting_ids.index)
        ].sort_values("rruff_id")
    )

## 3. Data integrity and spectral consistency

We investigate:
- Conflicting mineral labels associated with the same specimen.
- Consistency between filenames and metadata headers.
- Distribution of excitation wavelengths.
- Variation in spectral resolution and Raman shift ranges.

In [ ]:
suspect_ids = ["R050194", "R230009"]

suspect_rows = catalog[
    catalog["rruff_id"].isin(suspect_ids)
].sort_values(["rruff_id", "mineral", "filename"])

for _, row in suspect_rows.iterrows():

    metadata = read_rruff_metadata(Path(row["path"]))

    print("\n" + "=" * 80)
    print("Filename:", row["filename"])
    print("Mineral from filename:", row["mineral"])
    print("RRUFF ID from filename:", row["rruff_id"])

    print("Mineral from header:", metadata.get("NAMES"))
    print("RRUFF ID from header:", metadata.get("RRUFFID"))
    print("Identification status:", metadata.get("STATUS"))
    print("Laser wavelength:", metadata.get("RAMAN WAVELENGTH"))
    print("URL:", metadata.get("URL"))

In [ ]:
laser_summary = (
    catalog
    .groupby("laser_nm", dropna=False)
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique"),
        n_minerals=("mineral", "nunique")
    )
    .sort_values("n_spectra", ascending=False)
)

print(laser_summary.reset_index().to_string(index=False))

print("\nTotal spectra:", laser_summary["n_spectra"].sum())

In [ ]:
unusual_wavelengths = catalog[
    catalog["laser_nm"].isna()
    | ~catalog["laser_nm"].isin(
        ["532", "780", "785", "514", "633", "632", "458", "515"]
    )
]

display(
    unusual_wavelengths[
        ["mineral", "rruff_id", "filename", "laser_nm"]
    ].to_string(index=False)
)

In [ ]:
rng = np.random.default_rng(42)

sampled_files = rng.choice(
    processed_files,
    size=min(100, len(processed_files)),
    replace=False
)

grid_records = []

for file in sampled_files:

    try:
        spectrum = read_rruff_spectrum(file)

        x = spectrum["raman_shift_cm1"].to_numpy()
        differences = np.diff(x)

        grid_records.append({
            "filename": file.name,
            "n_points": len(x),
            "shift_min": x.min(),
            "shift_max": x.max(),
            "step_median": np.median(differences),
            "step_std": np.std(differences)
        })

    except Exception as error:
        print(f"Could not read {file.name}: {error}")

grid_stats = pd.DataFrame(grid_records)

display(
    grid_stats[
        ["n_points", "shift_min", "shift_max",
         "step_median", "step_std"]
    ].describe().round(4)
)

## 4. Building a metadata-validated dataset catalogue

The original filenames are used to identify potential metadata
inconsistencies.

Mineral labels and specimen identifiers are extracted from
the RRUFF file headers and used as the primary metadata.

We also retain the original filename fields for quality control.

In [ ]:
metadata_records = []
metadata_errors = []

for i, file in enumerate(processed_files):

    try:
        metadata = read_rruff_metadata(file)

        parts = file.name.split("__")

        record = {
            # Information from filename
            "filename_mineral": parts[0] if len(parts) > 0 else None,
            "filename_rruff_id": parts[1] if len(parts) > 1 else None,
            "filename_laser_token": parts[3] if len(parts) > 3 else None,

            # Information from metadata header
            "mineral": metadata.get("NAMES"),
            "rruff_id": metadata.get("RRUFFID"),
            "laser_wavelength_raw": metadata.get("RAMAN WAVELENGTH"),
            "status": metadata.get("STATUS"),
            "filetype": metadata.get("FILETYPE"),

            # File information
            "filename": file.name,
            "path": str(file)
        }

        metadata_records.append(record)

    except Exception as error:

        metadata_errors.append({
            "filename": file.name,
            "error": str(error)
        })

metadata_catalog = pd.DataFrame(metadata_records)

print("Successfully processed:", len(metadata_catalog))
print("Metadata errors:", len(metadata_errors))

display(metadata_catalog.head())

In [ ]:
metadata_catalog["mineral_name_match"] = (
    metadata_catalog["filename_mineral"].str.casefold()
    == metadata_catalog["mineral"].str.casefold()
)

metadata_catalog["rruff_id_match"] = (
    metadata_catalog["filename_rruff_id"]
    == metadata_catalog["rruff_id"]
)

metadata_catalog["laser_token_match"] = (
    metadata_catalog["filename_laser_token"]
    == metadata_catalog["laser_wavelength_raw"]
)

print("Mineral name mismatches:",
      (~metadata_catalog["mineral_name_match"]).sum())

print("RRUFF ID mismatches:",
      (~metadata_catalog["rruff_id_match"]).sum())

print("Laser token mismatches:",
      (~metadata_catalog["laser_token_match"]).sum())

In [ ]:
mismatches = metadata_catalog[
    (~metadata_catalog["mineral_name_match"])
    | (~metadata_catalog["rruff_id_match"])
]

display(
    mismatches[
        [
            "filename_mineral",
            "mineral",
            "filename_rruff_id",
            "rruff_id",
            "filename",
        ]
    ].to_string(index=False)
)

In [ ]:
print(
    "Total unique specimens:",
    metadata_catalog["rruff_id"].nunique()
)

labels_per_specimen = (
    metadata_catalog
    .dropna(subset=["rruff_id", "mineral"])
    .groupby("rruff_id")["mineral"]
    .nunique()
)

conflicting_specimens = labels_per_specimen[
    labels_per_specimen > 1
]

print(
    "Specimens associated with multiple mineral labels:",
    len(conflicting_specimens)
)

if len(conflicting_specimens) > 0:

    display(
        metadata_catalog[
            metadata_catalog["rruff_id"].isin(
                conflicting_specimens.index
            )
        ][
            ["rruff_id", "mineral", "filename"]
        ]
    )

In [ ]:
metadata_mineral_summary = (
    metadata_catalog
    .dropna(subset=["mineral", "rruff_id"])
    .groupby("mineral")
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique")
    )
    .sort_values("n_specimens", ascending=False)
)

display(metadata_mineral_summary.head(25))

In [ ]:
thresholds = [5, 8, 10, 12, 15, 20]

for threshold in thresholds:

    n_classes = (
        metadata_mineral_summary["n_specimens"] >= threshold
    ).sum()

    print(
        f"Classes with at least {threshold} specimens: "
        f"{n_classes}"
    )

In [ ]:
status_summary = (
    metadata_catalog
    .groupby("status", dropna=False)
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique")
    )
    .sort_values("n_spectra", ascending=False)
)

display(status_summary)

In [ ]:
for status in metadata_catalog["status"].dropna().unique():

    print("\n" + "=" * 70)
    print("STATUS:", status)

    display(
        metadata_catalog[
            metadata_catalog["status"] == status
        ][
            ["mineral", "rruff_id", "filename"]
        ].head(3)
    )

In [ ]:
wavelength_summary = (
    metadata_catalog["laser_wavelength_raw"]
    .value_counts(dropna=False)
    .rename_axis("laser_wavelength_raw")
    .reset_index(name="n_spectra")
)

display(wavelength_summary)

In [ ]:
wavelength_comparison = (
    metadata_catalog
    .groupby(
        [
            "filename_laser_token",
            "laser_wavelength_raw"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_spectra")
    .sort_values("n_spectra", ascending=False)
)

display(wavelength_comparison.head(30))

In [ ]:
wavelength_specimen_summary = (
    metadata_catalog
    .dropna(subset=[
        "laser_wavelength_raw",
        "mineral",
        "rruff_id"
    ])
    .groupby("laser_wavelength_raw")
    .agg(
        n_spectra=("filename", "count"),
        n_specimens=("rruff_id", "nunique"),
        n_minerals=("mineral", "nunique")
    )
    .sort_values("n_spectra", ascending=False)
)

display(wavelength_specimen_summary)

In [ ]:
rng = np.random.default_rng(42)

sampled_files = rng.choice(
    processed_files,
    size=min(100, len(processed_files)),
    replace=False
)

grid_diagnostics = []

for file in sampled_files:

    spectrum = read_rruff_spectrum(file)

    x = spectrum["raman_shift_cm1"].to_numpy()
    differences = np.diff(x)

    positive_steps = differences[differences > 0]

    median_positive_step = (
        np.median(positive_steps)
        if len(positive_steps) > 0
        else np.nan
    )

    grid_diagnostics.append({
        "filename": file.name,
        "n_points": len(x),
        "shift_min": x.min(),
        "shift_max": x.max(),
        "step_median": np.median(differences),
        "step_std": np.std(differences),
        "duplicate_steps": np.sum(differences == 0),
        "negative_steps": np.sum(differences < 0),
        "max_gap": np.max(differences),
        "gap_ratio": (
            np.max(differences) / median_positive_step
            if median_positive_step > 0
            else np.nan
        )
    })

grid_diagnostics = pd.DataFrame(grid_diagnostics)

display(
    grid_diagnostics
    .sort_values("step_std", ascending=False)
    .head(10)
)

In [ ]:
spectral_ranges = [
    (100, 1000),
    (100, 1100),
    (100, 1200),
    (150, 1100),
    (150, 1200),
    (200, 1100),
    (200, 1200),
    (250, 1100),
    (250, 1200)
]

coverage_results = []

for lower, upper in spectral_ranges:

    covers_interval = (
        (grid_diagnostics["shift_min"] <= lower)
        &
        (grid_diagnostics["shift_max"] >= upper)
    )

    n_covered = covers_interval.sum()

    coverage_results.append({
        "lower_shift": lower,
        "upper_shift": upper,
        "n_spectra": n_covered,
        "coverage_pct": 100 * n_covered / len(grid_diagnostics)
    })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df.round(2))

## 5. Catalogue robust
Make the catalog more robust

In [ ]:
import re

def normalize_name(value):
    """Normalize labels for comparison, without changing original labels."""

    if pd.isna(value):
        return ""

    return re.sub(
        r"[\W_]+",
        "",
        str(value).casefold()
    )

metadata_catalog["normalized_name_match"] = (
    metadata_catalog["filename_mineral"].apply(normalize_name)
    == metadata_catalog["mineral"].apply(normalize_name)
)

metadata_catalog["name_mismatch_type"] = np.select(
    [
        metadata_catalog["normalized_name_match"],
        metadata_catalog["filename_mineral"].str.lower().isin(
            ["unknown", "altered"]
        )
    ],
    [
        "Formatting difference only",
        "Placeholder in filename"
    ],
    default="Other difference"
)

display(
    metadata_catalog["name_mismatch_type"]
    .value_counts()
)

In [ ]:
display(
    metadata_catalog[
        metadata_catalog["name_mismatch_type"] != "Formatting difference only"
    ][
        [
            "filename_mineral",
            "mineral",
            "name_mismatch_type",
            "rruff_id"
        ]
    ].to_string(index=False)
)

In [ ]:
def classify_id_relation(row):

    filename_id = str(row["filename_rruff_id"])
    header_id = str(row["rruff_id"])

    if filename_id == header_id:
        return "Exact match"

    if filename_id.lower() in ["unknown", "altered", "nan", ""]:
        return "Filename placeholder"

    # Some external reference IDs have an X prefix in the header
    if header_id == "X" + filename_id:
        return "Header adds X prefix"

    # Filename contains additional information after the header ID
    if filename_id.startswith(header_id):
        return "Header ID plus filename suffix"

    if header_id.startswith(filename_id):
        return "Filename ID is a header prefix"

    return "Unresolved discrepancy"


metadata_catalog["id_relation"] = metadata_catalog.apply(
    classify_id_relation,
    axis=1
)

display(
    metadata_catalog["id_relation"].value_counts()
)

In [ ]:
unresolved_ids = metadata_catalog[
    metadata_catalog["id_relation"] == "Unresolved discrepancy"
]

display(
    unresolved_ids[
        [
            "filename_mineral",
            "mineral",
            "filename_rruff_id",
            "rruff_id",
            "status",
            "filename"
        ]
    ].to_string(index=False)
)

In [ ]:
def classify_status(status):

    if pd.isna(status):
        return "Missing"

    s = status.casefold()

    if "not yet confirmed" in s or "unconfirmed" in s:
        return "Unconfirmed"

    if "diffraction" in s:
        return "Diffraction-supported"

    if "only by raman spectroscopy" in s:
        return "Raman-only"

    if "raman spectroscopy" in s and "chemical analysis" in s:
        return "Raman + chemical analysis"

    if "chemical analysis" in s:
        return "Chemical analysis only"

    return "Other"


metadata_catalog["status_tier"] = (
    metadata_catalog["status"].apply(classify_status)
)

status_tier_summary = (
    metadata_catalog
    .groupby("status_tier")
    .agg(
        n_spectra=("filename", "count"),
        n_sample_ids=("rruff_id", "nunique"),
        n_minerals=("mineral", "nunique")
    )
    .sort_values("n_spectra", ascending=False)
)

display(status_tier_summary)

In [ ]:
status_consistency = (
    metadata_catalog
    .groupby("rruff_id")
    .agg(
        n_statuses=("status", "nunique"),
        statuses=("status", lambda x: list(x.dropna().unique())),
        mineral=("mineral", "first")
    )
)

mixed_status_ids = status_consistency[
    status_consistency["n_statuses"] > 1
]

print("Sample IDs with multiple statuses:", len(mixed_status_ids))

display(mixed_status_ids)

In [ ]:
def parse_wavelength(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().casefold()

    if value in ["", "unknown"]:
        return np.nan

    match = re.search(r"\d+(?:\.\d+)?", value)

    if match:
        return float(match.group())

    return np.nan


metadata_catalog["wavelength_nm"] = (
    metadata_catalog["laser_wavelength_raw"]
    .apply(parse_wavelength)
)

display(
    metadata_catalog["wavelength_nm"]
    .value_counts(dropna=False)
)

### 5.1 How much data for the experiment

In [ ]:
data_532 = metadata_catalog[
    metadata_catalog["wavelength_nm"] == 532
].copy()

print("Total 532 nm spectra:", len(data_532))
print(
    "Unique RRUFF sample IDs:",
    data_532["rruff_id"].nunique()
)

In [ ]:
data_532_confirmed = data_532[
    data_532["status_tier"] == "Diffraction-supported"
].copy()

print(
    "Diffraction-supported 532 nm spectra:",
    len(data_532_confirmed)
)

print(
    "Unique sample IDs:",
    data_532_confirmed["rruff_id"].nunique()
)

In [ ]:
confirmed_mineral_summary = (
    data_532_confirmed
    .groupby("mineral")
    .agg(
        n_spectra=("filename", "count"),
        n_sample_ids=("rruff_id", "nunique")
    )
    .sort_values("n_sample_ids", ascending=False)
)

display(confirmed_mineral_summary.head(30))

In [ ]:
thresholds = [5, 6, 8, 10, 12, 15]

for threshold in thresholds:

    n_classes = (
        confirmed_mineral_summary["n_sample_ids"]
        >= threshold
    ).sum()

    print(
        f"Classes with at least {threshold} sample IDs: "
        f"{n_classes}"
    )

### 5.2 The unusual Heflikite spectrum

In [ ]:
heflikite_file = next(
    file for file in processed_files
    if file.name.startswith("Heflikite__R250030")
)

heflikite_spectrum = read_rruff_spectrum(heflikite_file)

x = heflikite_spectrum["raman_shift_cm1"].to_numpy()
y = heflikite_spectrum["intensity"].to_numpy()

print("Total rows:", len(x))
print("Unique Raman shifts:", len(np.unique(x)))
print("Duplicate Raman-shift rows:",
      heflikite_spectrum["raman_shift_cm1"].duplicated().sum())

print("\nMost frequent Raman shifts:")

display(
    heflikite_spectrum["raman_shift_cm1"]
    .value_counts()
    .head(15)
)

In [ ]:
duplicate_positions = heflikite_spectrum[
    heflikite_spectrum["raman_shift_cm1"].duplicated(
        keep=False
    )
].sort_values("raman_shift_cm1")

display(duplicate_positions.head(30))

In [ ]:
with open(
    heflikite_file,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:

    numerical_lines = [
        line.strip()
        for line in f
        if line.strip() and not line.startswith("#")
    ]

print("\nFirst 30 numerical lines:")

for line in numerical_lines[:30]:
    print(line)

## 6. Creation of Dataset


In [ ]:
candidate_minerals = [
    "Grossular",
    "Andradite",
    "Diopside",
    "Epidote",
    "Forsterite",
    "Enstatite",
    "Elbaite",
    "Beryl",
    "Fluorapatite",
    "Almandine",
    "Pyrope",
    "Wendwilsonite"
]

In [ ]:
# Sample IDs with inconsistent identification status
mixed_status_ids = set(
    status_consistency[
        status_consistency["n_statuses"] > 1
    ].index
)

# Unresolved filename/header identifier discrepancies
unresolved_sample_ids = set(
    unresolved_ids["rruff_id"].dropna()
)

# Combined quarantine list
quarantined_ids = mixed_status_ids | unresolved_sample_ids

print("Sample IDs quarantined:", len(quarantined_ids))
print("IDs:", sorted(quarantined_ids))

In [ ]:
candidate_data = metadata_catalog[
    (metadata_catalog["status_tier"] == "Diffraction-supported")
    & (metadata_catalog["mineral"].isin(candidate_minerals))
    & (metadata_catalog["rruff_id"].notna())
    & (~metadata_catalog["rruff_id"].isin(quarantined_ids))
].copy()

print("Total candidate spectra:", len(candidate_data))

print(
    "Unique candidate sample IDs:",
    candidate_data["rruff_id"].nunique()
)

In [ ]:
# Summary using all available wavelengths
all_wavelength_summary = (
    candidate_data
    .groupby("mineral")
    .agg(
        n_spectra_all=("filename", "count"),
        n_samples_all=("rruff_id", "nunique")
    )
)

# Summary using only 532 nm
summary_532 = (
    candidate_data[
        candidate_data["wavelength_nm"] == 532
    ]
    .groupby("mineral")
    .agg(
        n_spectra_532=("filename", "count"),
        n_samples_532=("rruff_id", "nunique")
    )
)

comparison = (
    all_wavelength_summary
    .join(summary_532)
    .fillna(0)
)

comparison["additional_sample_ids"] = (
    comparison["n_samples_all"]
    - comparison["n_samples_532"]
)

display(comparison.sort_values(
    "n_samples_all",
    ascending=False
))


In [ ]:
wavelength_composition = (
    candidate_data
    .groupby(["mineral", "wavelength_nm"])["rruff_id"]
    .nunique()
    .unstack(fill_value=0)
)

display(wavelength_composition)

In [ ]:
uranoclite_records = metadata_catalog[
    metadata_catalog["rruff_id"] == "R250144"
][
    [
        "mineral",
        "rruff_id",
        "filename",
        "wavelength_nm",
        "status",
        "status_tier"
    ]
]

display(uranoclite_records.to_string(index=False))

### 6.1 Investigate the Heflikite

In [ ]:
shift_counts = (
    heflikite_spectrum
    .groupby("raman_shift_cm1")
    .size()
)

count_distribution = (
    shift_counts
    .value_counts()
    .sort_index()
    .rename_axis("observations_per_shift")
    .reset_index(name="n_shift_positions")
)

display(count_distribution)

In [ ]:
heflikite_inspection = heflikite_spectrum.copy()

heflikite_inspection["occurrence"] = (
    heflikite_inspection
    .groupby("raman_shift_cm1")
    .cumcount()
)

heflikite_pivot = (
    heflikite_inspection
    .pivot(
        index="raman_shift_cm1",
        columns="occurrence",
        values="intensity"
    )
)

ax = heflikite_pivot.plot(
    figsize=(12, 5),
    linewidth=1,
    alpha=0.8
)

ax.set_xlabel("Raman shift (cm$^{-1}$)")
ax.set_ylabel("Processed intensity")
ax.set_title("Heflikite: repeated observations at identical Raman shifts")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
heflikite_metadata = read_rruff_metadata(heflikite_file)

for key, value in heflikite_metadata.items():

    print(f"{key}: {value}")

## 7. Construction of the primary classification dataset

The primary dataset contains 12 selected mineral species,
measured at 532 nm, with RRUFF metadata indicating
diffraction-based identification evidence.

The header RRUFF ID is used as the sample grouping identifier.

In [ ]:
# Selected mineral species
candidate_minerals = [
    "Grossular",
    "Andradite",
    "Diopside",
    "Epidote",
    "Forsterite",
    "Enstatite",
    "Elbaite",
    "Beryl",
    "Fluorapatite",
    "Almandine",
    "Pyrope",
    "Wendwilsonite"
]

# Identify sample IDs with genuinely inconsistent
# identification evidence categories.
evidence_counts = (
    metadata_catalog
    .groupby("rruff_id")["status_tier"]
    .nunique()
)

mixed_evidence_ids = set(
    evidence_counts[evidence_counts > 1].index
)

# Exclude unresolved filename/header ID discrepancies.
unresolved_sample_ids = set(
    unresolved_ids["rruff_id"].dropna()
)

# Construct the primary dataset.
primary_data = metadata_catalog[
    (metadata_catalog["wavelength_nm"] == 532)
    & (metadata_catalog["mineral"].isin(candidate_minerals))
    & (metadata_catalog["status_tier"] == "Diffraction-supported")
    & (metadata_catalog["rruff_id"].notna())
    & (~metadata_catalog["rruff_id"].isin(unresolved_sample_ids))
    & (~metadata_catalog["rruff_id"].isin(mixed_evidence_ids))
].copy()

primary_data = primary_data.reset_index(drop=True)

print("Total spectra:", len(primary_data))
print("Unique sample IDs:", primary_data["rruff_id"].nunique())
print("Unique minerals:", primary_data["mineral"].nunique())

In [ ]:
# Class distribution
final_class_summary = (
    primary_data
    .groupby("mineral")
    .agg(
        n_spectra=("filename", "count"),
        n_sample_ids=("rruff_id", "nunique")
    )
    .sort_values("n_sample_ids", ascending=False)
)

display(final_class_summary)

# Check duplicate sample IDs
duplicate_sample_ids = (
    primary_data["rruff_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("\nSample IDs with multiple 532 nm spectra:")
display(duplicate_sample_ids)

# Check wavelength consistency
assert primary_data["wavelength_nm"].eq(532).all()

# Check that every selected class is represented
assert set(primary_data["mineral"]) == set(candidate_minerals)

# Check that sample IDs are not duplicated
assert duplicate_sample_ids.empty

# Final validation
assert primary_data["rruff_id"].nunique() == 137

print("\nAll primary dataset validation checks passed.")

In [ ]:
manifest_dir = PROJECT_ROOT / "data" / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest = primary_data[
    [
        "mineral",
        "rruff_id",
        "wavelength_nm",
        "status",
        "filetype",
        "filename",
        "path"
    ]
].copy()

# Store relative paths rather than machine-specific
# absolute Windows paths.
manifest["relative_path"] = manifest["path"].apply(
    lambda p: Path(p).relative_to(DATA_DIR).as_posix()
)

manifest = manifest.drop(columns=["path"])

manifest_path = (
    manifest_dir / "primary_532nm_manifest.csv"
)

manifest.to_csv(manifest_path, index=False)

print("Manifest saved to:", manifest_path)
print("Number of records:", len(manifest))

## 8. Spectral characteristics of the primary dataset

This section investigates the spectral ranges, sampling
resolution, duplicate Raman-shift positions and intensity
characteristics of the selected 137 sample-ID groups.

In [ ]:
spectral_records = []

for row in primary_data.itertuples(index=False):

    spectrum = read_rruff_spectrum(row.path)

    x = spectrum["raman_shift_cm1"].to_numpy()
    y = spectrum["intensity"].to_numpy()

    # Unique Raman-shift positions
    unique_x = np.unique(x)

    # Differences between consecutive original observations
    original_steps = np.diff(x)

    # Differences between unique sorted positions
    unique_steps = np.diff(unique_x)

    spectral_records.append({

        "mineral": row.mineral,
        "rruff_id": row.rruff_id,
        "filename": row.filename,

        "n_points": len(x),
        "n_unique_shifts": len(unique_x),
        "n_duplicate_rows": len(x) - len(unique_x),

        "shift_min": x.min(),
        "shift_max": x.max(),

        "strictly_increasing": np.all(original_steps > 0),
        "nondecreasing": np.all(original_steps >= 0),

        "median_step": np.median(unique_steps),
        "step_std": np.std(unique_steps),

        "min_intensity": y.min(),
        "max_intensity": y.max(),

        "negative_intensity_count": np.sum(y < 0),
        "finite_values": np.isfinite(x).all() and np.isfinite(y).all()
    })

spectral_audit = pd.DataFrame(spectral_records)

print("Audited spectra:", len(spectral_audit))
display(spectral_audit.head())

In [ ]:
display(
    spectral_audit[
        ["n_points", "shift_min", "shift_max",
         "median_step", "step_std", "min_intensity",
         "max_intensity"]
    ].describe().round(4)
)

In [ ]:
common_lower = spectral_audit["shift_min"].max()
common_upper = spectral_audit["shift_max"].min()

print("Common lower boundary:", common_lower)
print("Common upper boundary:", common_upper)
print("Common spectral width:", common_upper - common_lower)

In [ ]:
candidate_ranges = [
    (100, 1000),
    (100, 1100),
    (150, 1100),
    (200, 1100),
    (200, 1200),
    (250, 1100),
    (250, 1200)
]

coverage_records = []

for lower, upper in candidate_ranges:

    covered = (
        (spectral_audit["shift_min"] <= lower)
        &
        (spectral_audit["shift_max"] >= upper)
    )

    coverage_records.append({
        "lower_shift": lower,
        "upper_shift": upper,
        "n_covered": int(covered.sum()),
        "coverage_pct": 100 * covered.mean()
    })

coverage_summary = pd.DataFrame(coverage_records)

display(coverage_summary.round(2))

In [ ]:
class_range_summary = (
    spectral_audit
    .groupby("mineral")
    .agg(
        n_sample_ids=("rruff_id", "nunique"),
        lowest_shift=("shift_min", "min"),
        lower_q05=("shift_min", lambda x: x.quantile(0.05)),
        median_start=("shift_min", "median"),
        median_end=("shift_max", "median"),
        upper_q05=("shift_max", lambda x: x.quantile(0.05)),
        lowest_max_shift=("shift_max", "min")
    )
)

display(class_range_summary.round(2))

In [ ]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

spectral_audit.to_csv(
    reports_dir / "primary_spectral_audit.csv",
    index=False
)

coverage_summary.to_csv(
    reports_dir / "spectral_coverage_summary.csv",
    index=False
)

print("Spectral audit files saved.")

### 8.1 Generate actual splits

In [ ]:
from mineral_id.splits import make_grouped_splits

splits = make_grouped_splits(
    primary_data,
    group_col="rruff_id",
    label_col="mineral",
    train_fraction=0.60,
    calibration_fraction=0.20,
    test_fraction=0.20,
    random_state=42
)

In [ ]:
for name, split_data in splits.items():

    print(f"\n{'=' * 60}")
    print(name.upper())

    print("Number of spectra:", len(split_data))
    print(
        "Number of sample IDs:",
        split_data["rruff_id"].nunique()
    )

    display(
        split_data.groupby("mineral")["rruff_id"]
        .nunique()
    )

In [ ]:
split_assignment = pd.concat(
    [
        split_data[
            ["rruff_id", "mineral"]
        ]
        .drop_duplicates()
        .assign(split=name)

        for name, split_data in splits.items()
    ],
    ignore_index=True
)

split_assignment = split_assignment.sort_values(
    ["mineral", "rruff_id"]
).reset_index(drop=True)

assignment_path = (
    PROJECT_ROOT
    / "data"
    / "manifests"
    / "primary_532nm_split_assignment.csv"
)

split_assignment.to_csv(
    assignment_path,
    index=False
)

print("Split assignment saved:", assignment_path)

## 9. Spectral preprocessing and feature construction

All selected Raman spectra are interpolated onto a common
wavenumber grid.

We investigate the resulting feature representation and
compare different intensity normalisation techniques.

In [ ]:
from mineral_id.preprocessing import (
    make_raman_grid,
    interpolate_spectrum,
    integrated_area,
    normalize_spectrum,
    preprocess_spectrum
)

In [ ]:
RAMAN_START = 205.0
RAMAN_STOP = 1200.0
RAMAN_STEP = 0.5

grid = make_raman_grid(
    start=RAMAN_START,
    stop=RAMAN_STOP,
    step=RAMAN_STEP
)

print("Number of grid points:", len(grid))
print("First position:", grid[0])
print("Last position:", grid[-1])
print("Grid spacing:", np.diff(grid).mean())

In [ ]:
coverage_check = (
    (spectral_audit["shift_min"] <= RAMAN_START)
    &
    (spectral_audit["shift_max"] >= RAMAN_STOP)
)

print("Spectra covering the complete grid:", coverage_check.sum())

assert coverage_check.all()

In [ ]:
X_raw_list = []

for row in primary_data.itertuples(index=False):

    spectrum = read_rruff_spectrum(row.path)

    interpolated = interpolate_spectrum(
        spectrum=spectrum,
        grid=grid
    )

    X_raw_list.append(interpolated)


X_raw = np.vstack(X_raw_list)

feature_metadata = primary_data[
    ["mineral", "rruff_id", "filename"]
].reset_index(drop=True)


print("Feature matrix shape:", X_raw.shape)

print("All finite:", np.isfinite(X_raw).all())

print("Negative intensity values:", (X_raw < 0).sum())

print(
    "Spectra containing negative values:",
    (X_raw < 0).any(axis=1).sum()
)

In [ ]:
processed_dir = (
    PROJECT_ROOT / "data" / "processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    processed_dir / "primary_532nm_interpolated.npz",

    X_raw=X_raw,

    raman_grid=grid,

    mineral=feature_metadata["mineral"].to_numpy(dtype=str),

    rruff_id=feature_metadata["rruff_id"].to_numpy(dtype=str)
)

print("Interpolated dataset saved.")

## 10. Compare normalisation techniques

In [ ]:
max_scales = np.max(X_raw, axis=1)

l2_scales = np.linalg.norm(
    X_raw,
    axis=1
)

area_scales = np.array([
    integrated_area(spectrum, grid)
    for spectrum in X_raw
])

normalization_diagnostics = pd.DataFrame({

    "mineral": feature_metadata["mineral"],

    "rruff_id": feature_metadata["rruff_id"],

    "max_scale": max_scales,

    "l2_scale": l2_scales,

    "area_scale": area_scales

})

display(
    normalization_diagnostics[
        ["max_scale", "l2_scale", "area_scale"]
    ].describe().round(4)
)

In [ ]:
print(
    "Non-positive integrated areas:",
    (area_scales <= 1e-12).sum()
)

In [ ]:
X_max = np.vstack([
    normalize_spectrum(x, method="max")
    for x in X_raw
])

X_l2 = np.vstack([
    normalize_spectrum(x, method="l2")
    for x in X_raw
])

if np.all(area_scales > 1e-12):

    X_area = np.vstack([
        normalize_spectrum(
            x,
            method="area",
            grid=grid
        )
        for x in X_raw
    ])

else:

    X_area = None

    print(
        "Area normalisation unavailable for every sample. "
        "We will not use it in the primary comparison."
    )

In [ ]:
for name, matrix in {
    "Maximum": X_max,
    "L2": X_l2
}.items():

    print(f"\n{name} normalization")

    print("Shape:", matrix.shape)

    print("All finite:", np.isfinite(matrix).all())

    print("Negative values:", (matrix < 0).sum())


if X_area is not None:

    print("\nArea normalization")

    print("Shape:", X_area.shape)

    print(
        "All finite:",
        np.isfinite(X_area).all()
    )

In [ ]:
sample_index = 0

fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 8),
    sharex=True
)

# Original interpolated spectrum
axes[0].plot(
    grid,
    X_raw[sample_index],
    linewidth=1
)

axes[0].set_title("Original interpolated spectrum")
axes[0].set_ylabel("Intensity")

# Normalised spectra
axes[1].plot(
    grid,
    X_max[sample_index],
    label="Maximum",
    linewidth=1
)

axes[1].plot(
    grid,
    X_l2[sample_index],
    label="L2",
    linewidth=1
)

if X_area is not None:

    axes[1].plot(
        grid,
        X_area[sample_index],
        label="Area",
        linewidth=1
    )

axes[1].set_title("Comparison of normalisation methods")
axes[1].set_xlabel("Raman shift (cm$^{-1}$)")
axes[1].set_ylabel("Normalised intensity")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
print("X_area shape:", X_area.shape)
print("All finite:", np.isfinite(X_area).all())
print(
    "Minimum signed area after normalisation:",
    min(integrated_area(x, grid) for x in X_area)
)
print(
    "Maximum signed area after normalisation:",
    max(integrated_area(x, grid) for x in X_area)
)

In [ ]:
np.savez_compressed(
    PROJECT_ROOT / "data" / "processed" /
    "primary_532nm_features.npz",

    X_raw=X_raw,
    X_max=X_max,
    X_l2=X_l2,
    X_area=X_area,

    raman_grid=grid,

    mineral=feature_metadata["mineral"].to_numpy(dtype=str),

    rruff_id=feature_metadata["rruff_id"].to_numpy(dtype=str)
)

## 11. Exploratory analysis of mineral spectral signatures

We investigate the spectral characteristics of each mineral
using the L2-normalised training spectra.

The calibration and test partitions remain untouched.

In [ ]:
# Identify sample IDs belonging to the training partition
train_ids = set(splits["train"]["rruff_id"])

# Preserve the original feature matrix ordering
train_mask = (
    feature_metadata["rruff_id"]
    .isin(train_ids)
    .to_numpy()
)

# Construct training-only feature matrix
X_train_l2 = X_l2[train_mask]

# Corresponding metadata
meta_train = (
    feature_metadata.loc[train_mask]
    .reset_index(drop=True)
)

print("Training feature matrix:", X_train_l2.shape)
print("Training observations:", len(meta_train))
print("Training sample IDs:", meta_train["rruff_id"].nunique())
print("Training mineral classes:", meta_train["mineral"].nunique())

In [ ]:
assert X_train_l2.shape[0] == len(meta_train)

assert X_train_l2.shape[1] == len(grid)

assert meta_train["rruff_id"].is_unique

assert set(meta_train["rruff_id"]) == train_ids

assert np.isfinite(X_train_l2).all()

In [ ]:
mineral_order = [
    "Grossular",
    "Andradite",
    "Diopside",
    "Epidote",
    "Forsterite",
    "Enstatite",
    "Elbaite",
    "Beryl",
    "Fluorapatite",
    "Almandine",
    "Pyrope",
    "Wendwilsonite"
]

fig, axes = plt.subplots(
    3,
    4,
    figsize=(19, 12),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, mineral in zip(axes, mineral_order):

    mask = meta_train["mineral"].to_numpy() == mineral

    spectra = X_train_l2[mask]

    # Individual spectra
    for spectrum in spectra:

        ax.plot(
            grid,
            spectrum,
            color="grey",
            alpha=0.30,
            linewidth=0.65
        )

    # Class-level statistics
    median = np.median(spectra, axis=0)

    q25 = np.quantile(spectra, 0.25, axis=0)

    q75 = np.quantile(spectra, 0.75, axis=0)

    # Interquartile region
    ax.fill_between(
        grid,
        q25,
        q75,
        alpha=0.20,
        label="25th–75th percentile"
    )

    # Median spectral signature
    ax.plot(
        grid,
        median,
        linewidth=1.5,
        label="Median"
    )

    ax.set_title(
        f"{mineral} (n={len(spectra)})"
    )

    ax.grid(alpha=0.2)

    ax.set_xlim(205, 1200)


fig.supxlabel("Raman shift (cm$^{-1}$)", fontsize=12)

fig.supylabel("L2-normalised intensity", fontsize=12)

fig.suptitle(
    "Raman spectral signatures of the 12 selected minerals",
    fontsize=15,
    y=1.01
)

plt.tight_layout()

fig.savefig(
    PROJECT_ROOT / "reports" / "figures" /
    "mineral_spectral_signatures.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
garnet_group = [
    "Grossular",
    "Andradite",
    "Almandine",
    "Pyrope"
]

fig, ax = plt.subplots(figsize=(14, 6))

for mineral in garnet_group:

    mask = meta_train["mineral"].to_numpy() == mineral

    spectra = X_train_l2[mask]

    median = np.median(spectra, axis=0)

    ax.plot(
        grid,
        median,
        linewidth=1.5,
        label=f"{mineral} (n={len(spectra)})"
    )

ax.set_title("Comparison of garnet-group mineral spectra")

ax.set_xlabel("Raman shift (cm$^{-1}$)")

ax.set_ylabel("L2-normalised intensity")

ax.legend()

ax.grid(alpha=0.25)

plt.tight_layout()

fig.savefig(
    PROJECT_ROOT / "reports" / "figures" /
    "garnet_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
comparison_minerals = [
    "Forsterite",
    "Enstatite"
]

fig, ax = plt.subplots(figsize=(14, 6))

for mineral in comparison_minerals:

    mask = meta_train["mineral"].to_numpy() == mineral

    spectra = X_train_l2[mask]

    median = np.median(spectra, axis=0)

    ax.plot(
        grid,
        median,
        linewidth=1.6,
        label=f"{mineral} (n={len(spectra)})"
    )

ax.set_title("Forsterite versus Enstatite")

ax.set_xlabel("Raman shift (cm$^{-1}$)")

ax.set_ylabel("L2-normalised intensity")

ax.legend()

ax.grid(alpha=0.25)

plt.tight_layout()

plt.show()

In [ ]:
comparison_minerals = [
    "Beryl",
    "Elbaite"
]

fig, ax = plt.subplots(figsize=(14, 6))

for mineral in comparison_minerals:

    mask = meta_train["mineral"].to_numpy() == mineral

    spectra = X_train_l2[mask]

    median = np.median(spectra, axis=0)

    ax.plot(
        grid,
        median,
        linewidth=1.6,
        label=f"{mineral} (n={len(spectra)})"
    )

ax.set_title("Beryl versus Elbaite")

ax.set_xlabel("Raman shift (cm$^{-1}$)")

ax.set_ylabel("L2-normalised intensity")

ax.legend()

ax.grid(alpha=0.25)

plt.tight_layout()

plt.show()

### 11.1 Investigate the three spectra containing negative values

In [ ]:
negative_rows = np.flatnonzero(
    np.any(X_raw < 0, axis=1)
)

negative_summary = []

for i in negative_rows:

    negative_positions = grid[X_raw[i] < 0]

    negative_summary.append({
        "mineral": feature_metadata.iloc[i]["mineral"],

        "rruff_id": feature_metadata.iloc[i]["rruff_id"],

        "n_negative_values": int(
            np.sum(X_raw[i] < 0)
        ),

        "minimum_intensity": float(
            X_raw[i].min()
        ),

        "negative_shift_min": float(
            negative_positions.min()
        ),

        "negative_shift_max": float(
            negative_positions.max()
        )
    })

negative_summary = pd.DataFrame(negative_summary)

display(negative_summary)

In [ ]:
for idx in negative_rows:

    mineral = feature_metadata.iloc[idx]["mineral"]
    rruff_id = feature_metadata.iloc[idx]["rruff_id"]

    spectrum = X_raw[idx]

    negative_mask = spectrum < 0

    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(
        grid,
        spectrum,
        linewidth=0.8,
        label="Interpolated spectrum"
    )

    ax.scatter(
        grid[negative_mask],
        spectrum[negative_mask],
        label="Negative observations",
        s=18,
        zorder=3
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=0.8
    )

    ax.set_title(
        f"{mineral} — {rruff_id}"
    )

    ax.set_xlabel("Raman shift (cm$^{-1}$)")
    ax.set_ylabel("Original interpolated intensity")

    ax.legend()
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## 12. Pairwise mineral spectral similarity

We calculate the median L2-normalised spectral signature
of each mineral using training observations only.

Pairwise cosine similarity is used to quantify similarities
between mineral spectral profiles.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class_profiles = {}

for mineral in mineral_order:

    mask = (
        meta_train["mineral"].to_numpy() == mineral
    )

    spectra = X_train_l2[mask]

    profile = np.median(spectra, axis=0)

    # Normalise each median profile
    profile_norm = np.linalg.norm(profile)

    if profile_norm <= 1e-12:
        raise ValueError(
            f"Invalid median spectral profile: {mineral}"
        )

    class_profiles[mineral] = profile / profile_norm

# Stack all class profiles
profile_matrix = np.vstack([
    class_profiles[mineral]
    for mineral in mineral_order
])

print("Profile matrix:", profile_matrix.shape)

In [ ]:
similarity_matrix = cosine_similarity(
    profile_matrix
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=mineral_order,
    columns=mineral_order
)

display(similarity_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

im = ax.imshow(
    similarity_matrix,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(range(len(mineral_order)))
ax.set_yticks(range(len(mineral_order)))

ax.set_xticklabels(
    mineral_order,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(mineral_order)

for i in range(len(mineral_order)):

    for j in range(len(mineral_order)):

        ax.text(
            j,
            i,
            f"{similarity_matrix[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

ax.set_title(
    "Pairwise cosine similarity of mineral spectral signatures"
)

fig.colorbar(
    im,
    ax=ax,
    label="Cosine similarity"
)

plt.tight_layout()

fig.savefig(
    PROJECT_ROOT / "reports" / "figures" /
    "mineral_similarity_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()